# FIFA 2026 World Cup Match Predictor

A machine learning model to predict match outcomes for the 2026 FIFA World Cup.
Built using historical international football results from 1872 to present.

## 1. Load Data

In [134]:
import pandas as pd

df = pd.read_csv('data/results.csv')
print(df.shape)
print(df.dtypes)
df.head(10)

(49287, 9)
date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


## 2. Filter Data

Narrowing the dataset down to matches relevant to the 2026 World Cup:
- **1990 - Today** — modern football is tactically different from earlier eras, also exclude matches that have yet to happen
- **Competitive matches only** — friendlies don't reflect true team performance

In [135]:
# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])

# Filter 1: Only keep matches from 1990 onwards and before today
df = df[(df['date'] >= '1990-01-01') & (df['date'] < pd.Timestamp.today())]

# Filter 2: Drop friendlies
competitive = df[df['tournament'] != 'Friendly']

print(competitive.shape)
print(competitive['tournament'].value_counts())

(21404, 9)
tournament
FIFA World Cup qualification            6977
UEFA Euro qualification                 2114
African Cup of Nations qualification    1865
UEFA Nations League                      658
African Cup of Nations                   642
                                        ... 
TIFOCO Tournament                          1
Copa Confraternidad                        1
Benedikt Fontana Cup                       1
ConIFA Challenger Cup                      1
South Asian Super Cup                      1
Name: count, Length: 141, dtype: int64


## 3. Add Target Variable

Adding the `result` column — the value the model will learn to predict:
- **home_win** — home team scored more goals
- **away_win** — away team scored more goals
- **draw** — both teams scored the same

In [136]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

competitive['result'] = competitive.apply(get_result, axis=1)
print(competitive['result'].value_counts())

result
home_win    10478
away_win     6285
draw         4641
Name: count, dtype: int64


## 4. Feature Engineering

Building features for the model to learn from. 

## 4.1 Last 10 Matches

`get_form_last_10` calculates a team's win rate over their last 10 competitive matches prior to each game. Draws count as 0.5 to reflect the real points system. Returns 0.5 as a neutral default if a team has no prior match history.

In [137]:
def get_form_last_10(team, date, df, n=10):
    # Get all matches for this team before this date
    team_matches = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < date)].sort_values('date').tail(n)    
    
    if len(team_matches) == 0:
        return 0.5  # Neutral form if no match history

    wins = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            if row['result'] == 'home_win':
                wins += 1
            elif row['result'] == 'draw':
                wins += 0.5
        else:  # away team
            if row['result'] == 'away_win':
                wins += 1
            elif row['result'] == 'draw':
                wins += 0.5

    return wins / len(team_matches)  # Return the win percentage

# Apply to every match
competitive['home_form_last_10'] = competitive.apply(
    lambda row: get_form_last_10(row['home_team'], row['date'], competitive), axis=1
)
competitive['away_form_last_10'] = competitive.apply(
    lambda row: get_form_last_10(row['away_team'], row['date'], competitive), axis=1
)

print(competitive[['home_team', 'away_team', 'home_form_last_10', 'away_form_last_10', 'result']])

            home_team   away_team  home_form_last_10  away_form_last_10  \
17132        Thailand       Kenya               0.50               0.50   
17133        Colombia     Uruguay               0.50               0.50   
17134       Indonesia       Kenya               0.50               0.00   
17136   United States  Costa Rica               0.50               0.50   
17138      Costa Rica     Uruguay               1.00               1.00   
...               ...         ...                ...                ...   
49210          Kosovo      Turkey               0.80               0.75   
49211  Czech Republic     Denmark               0.70               0.65   
49212        Cameroon    China PR               0.50               0.40   
49213       Australia     Curaçao               0.85               0.55   
49214      Kazakhstan     Comoros               0.40               0.20   

         result  
17132  home_win  
17133  away_win  
17134  away_win  
17136  away_win  
17138  aw

## 4.2 Last 10 Goals Scored and Conceded Averages

`get_goal_avg_last_10` calculates each team's average goals scored and conceded over their last 10 competitive matches prior to each game. This captures both attacking and defensive strength independently of just wins and losses. Returns `(0.0, 0.0)` as a neutral default if a team has no prior match history.

In [138]:
def get_goal_avg_last_10(team, date, df, n=10):
    team_matches = df[((df['home_team'] == team) | (df['away_team'] == team)) & (df['date'] < date)].sort_values('date').tail(n)
    
    if len(team_matches) == 0:
        return 0.0, 0.0  # Neutral if no match history

    goals_scored = 0
    goals_conceded = 0
    for _, row in team_matches.iterrows():
        if row['home_team'] == team:
            goals_scored += row['home_score']
            goals_conceded += row['away_score']
        else:  # away team
            goals_scored += row['away_score']
            goals_conceded += row['home_score']
    
    return goals_scored / len(team_matches), goals_conceded / len(team_matches)

#Apply to every match
competitive[['home_goal_avg_last_10', 'home_concede_avg_last_10']] = competitive.apply(
    lambda row: get_goal_avg_last_10(row['home_team'], row['date'], competitive), axis=1, result_type='expand'
)
competitive[['away_goal_avg_last_10', 'away_concede_avg_last_10']] = competitive.apply(
    lambda row: get_goal_avg_last_10(row['away_team'], row['date'], competitive), axis=1, result_type='expand'
)

print(competitive[['home_team', 'away_team', 'home_goal_avg_last_10', 'home_concede_avg_last_10', 'away_goal_avg_last_10', 'away_concede_avg_last_10']])

            home_team   away_team  home_goal_avg_last_10  \
17132        Thailand       Kenya                    0.0   
17133        Colombia     Uruguay                    0.0   
17134       Indonesia       Kenya                    0.0   
17136   United States  Costa Rica                    0.0   
17138      Costa Rica     Uruguay                    2.0   
...               ...         ...                    ...   
49210          Kosovo      Turkey                    1.6   
49211  Czech Republic     Denmark                    2.2   
49212        Cameroon    China PR                    0.8   
49213       Australia     Curaçao                    1.8   
49214      Kazakhstan     Comoros                    1.1   

       home_concede_avg_last_10  away_goal_avg_last_10  \
17132                       0.0                    0.0   
17133                       0.0                    0.0   
17134                       0.0                    1.0   
17136                       0.0                

## 4.3 Last 10 Head to Head Results

`head_to_head_last_10` calculates each team's win rate across their last 10 meetings against each other prior to each game. Unlike recent form which looks at all matches, this captures patterns specific to this matchup — some teams consistently perform better or worse against certain opponents regardless of their overall form. Returns `(0.5, 0.5)` as a neutral default if the two teams have never met before.

In [139]:
def head_to_head_last_10(team1, team2, date, df, n=10):
    #Grabs every match between these two teams before this date, sorted by date, and takes the last 10
    matches = df[
        ((df['home_team'] == team1) & (df['away_team'] == team2)) |
        ((df['home_team'] == team2) & (df['away_team'] == team1))
    ]
    matches = matches[matches['date'] < date].sort_values('date').tail(n)

    # If no matches, return neutral 0.5/0.5
    if len(matches) == 0:
        return 0.5, 0.5

    #Iterate through these matches and count wins for each team
    team1_wins = 0
    team2_wins = 0

    for _, row in matches.iterrows():
        if row['home_team'] == team1:
            if row['result'] == 'home_win':
                team1_wins += 1
            elif row['result'] == 'away_win':
                team2_wins += 1
            else:
                team1_wins += 0.5
                team2_wins += 0.5
        else:  # team2 is home
            if row['result'] == 'home_win':
                team2_wins += 1
            elif row['result'] == 'away_win':
                team1_wins += 1
            else:
                team1_wins += 0.5
                team2_wins += 0.5

    total = len(matches)
    return team1_wins / total, team2_wins / total

competitive[['home_h2h_last_10', 'away_h2h_last_10']] = competitive.apply(
    lambda row: head_to_head_last_10(row['home_team'], row['away_team'], row['date'], competitive),
    axis=1, result_type='expand'
)

print(competitive[['home_team', 'away_team', 'home_h2h_last_10', 'away_h2h_last_10', 'result']].tail(50))

                    home_team                         away_team  \
49109                   Libya                             Niger   
49120                China PR                           Curaçao   
49121               Australia                          Cameroon   
49122              Azerbaijan                       Saint Lucia   
49123         Solomon Islands                          Bulgaria   
49124               Indonesia             Saint Kitts and Nevis   
49125                   Chile                        Cape Verde   
49126             New Zealand                           Finland   
49127                   Kenya                           Estonia   
49128                  Rwanda                           Grenada   
49129               Venezuela               Trinidad and Tobago   
49130              Uzbekistan                             Gabon   
49131                  Guyana                          Dominica   
49132                  Belize                      Sint Maarte

## 5 Final Filter

(~~Now that all features have been calculated using the full dataset, we filter down to only matches where both teams are among the 48 qualified World Cup 2026 teams. This gives us a training set that is directly relevant to the tournament while having benefited from the broader match history during feature engineering.~~)

Resetting the dataframe index to start from 0 after filtering and feature engineering.

In [140]:
# wc_teams = [
#     'United States', 'Canada', 'Mexico', 'Argentina', 'Brazil', 'Colombia',
#     'Ecuador', 'Paraguay', 'Uruguay', 'Australia', 'Iran', 'Japan', 'Jordan',
#     'South Korea', 'Qatar', 'Saudi Arabia', 'Uzbekistan', 'Iraq', 'Algeria',
#     'Cabo Verde', 'Ivory Coast', 'Egypt', 'Ghana', 'Morocco', 'Senegal',
#     'South Africa', 'Tunisia', 'DR Congo', 'Curacao', 'Haiti', 'Panama',
#     'New Zealand', 'England', 'France', 'Croatia', 'Norway', 'Portugal',
#     'Germany', 'Netherlands', 'Austria', 'Belgium', 'Scotland', 'Spain',
#     'Switzerland', 'Sweden', 'Turkey', 'Bosnia and Herzegovina', 'Czech Republic'
# ]

# competitive = competitive[
#     competitive['home_team'].isin(wc_teams) &
#     competitive['away_team'].isin(wc_teams)
# ].reset_index(drop=True)

# print(competitive.shape)
# print(competitive['result'].value_counts())
# competitive.tail(10)
competitive = competitive.reset_index(drop=True)

## 6. Encode Target Variable

Converting the `result` column from text to numbers since ML models require numerical inputs. `LabelEncoder` maps the three outcomes to: `0 = away_win`, `1 = draw`, `2 = home_win`.

In [141]:
from sklearn.preprocessing import LabelEncoder

# Encode result as a number
le = LabelEncoder()
competitive['result_encoded'] = le.fit_transform(competitive['result'])

print(le.classes_)  # shows what number maps to what result
competitive.head()

['away_win' 'draw' 'home_win']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_form_last_10,away_form_last_10,home_goal_avg_last_10,home_concede_avg_last_10,away_goal_avg_last_10,away_concede_avg_last_10,home_h2h_last_10,away_h2h_last_10,result_encoded
0,1990-01-29,Thailand,Kenya,2.0,1.0,King's Cup,Bangkok,Thailand,False,home_win,0.5,0.5,0.0,0.0,0.0,0.0,0.5,0.5,2
1,1990-02-02,Colombia,Uruguay,0.0,2.0,Miami Cup,Miami,United States,True,away_win,0.5,0.5,0.0,0.0,0.0,0.0,0.5,0.5,0
2,1990-02-02,Indonesia,Kenya,2.0,3.0,King's Cup,Bangkok,Thailand,True,away_win,0.5,0.0,0.0,0.0,1.0,2.0,0.5,0.5,0
3,1990-02-02,United States,Costa Rica,0.0,2.0,Miami Cup,Miami,United States,False,away_win,0.5,0.5,0.0,0.0,0.0,0.0,0.5,0.5,0
4,1990-02-04,Costa Rica,Uruguay,0.0,2.0,Miami Cup,Miami,United States,True,away_win,1.0,1.0,2.0,0.0,2.0,0.0,0.5,0.5,0


## 7. Train/Test Split

Splitting the data into training (80%) and testing (20%) sets. `shuffle=False` preserves chronological order so the model always trains on older matches and tests on more recent ones — this mirrors how the model will actually be used and prevents data leakage.

In [142]:
from sklearn.model_selection import train_test_split

features = [
    'home_form_last_10',
    'away_form_last_10',
    'home_goal_avg_last_10',
    'home_concede_avg_last_10',
    'away_goal_avg_last_10',
    'away_concede_avg_last_10',
    'home_h2h_last_10',
    'away_h2h_last_10',
    'neutral'
]

X = competitive[features]
y = competitive['result_encoded']

# Time-based split — train on older matches, test on recent ones
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 17123
Testing samples: 4281


## 8. Train the Model

Training an XGBoost classifier on the training set. Key parameters:
- **n_estimators** — number of decision trees to build
- **learning_rate** — how much each tree contributes to the final result, lower values are more conservative
- **max_depth** — how deep each tree can grow, higher values risk overfitting
- **random_state** — ensures reproducibility, the model will produce the same result every run

In [143]:
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    eval_metric='mlogloss',
    random_state=42
)

model.fit(X_train, y_train, sample_weight=sample_weights)
print("Model trained successfully")

Model trained successfully


## 9. Evaluate the Model

Evaluating the model on the test set using two metrics:
- **Accuracy** — percentage of matches where the model predicted the correct outcome
- **Log Loss** — measures how confident and correct the probability predictions are, lower is better

The classification report breaks down performance per outcome. The balanced class weights ensure the model learns all three outcomes rather than defaulting to predicting home wins.

In [144]:
from sklearn.metrics import accuracy_score, log_loss, classification_report

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_pred_proba)

print(f"Accuracy: {accuracy:.2%}")
print(f"Log Loss: {loss:.4f}")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 52.21%
Log Loss: 0.9478
              precision    recall  f1-score   support

    away_win       0.56      0.56      0.56      1315
        draw       0.27      0.38      0.32       953
    home_win       0.70      0.56      0.62      2013

    accuracy                           0.52      4281
   macro avg       0.51      0.50      0.50      4281
weighted avg       0.56      0.52      0.54      4281

